# Benchmark Analysis — OMERO-Screen Pipeline Performance

Comparing segmentation performance across hardware configurations and Cellpose versions.

**Plate 3802** | 4 images/run | 10 repeats per configuration

| Configuration | Hardware | Cellpose |
|---------------|----------|-----------|
| Mac CPU | Apple arm M1 (10-core CPU) | cp3 |
| Apple MPS | Apple arm M1 (Metal GPU) | cp3, cp4 |
| NVIDIA A40 | HPC node — Artemis | cp3, cp4 |
| RTX 5090 | Workstation | cp3, cp4 |

> **Note — GPU warm-up:** The first image per run includes model-loading overhead (~30–60 s on NVIDIA GPUs). Panels A, C, D exclude this first image to show steady-state performance.
>
> **Note — A40 nodes:** cp3 and cp4 were run on different nodes (artemis-a40-02 vs artemis-a40-10) with identical hardware specs; total-time differences may partly reflect network variation.

In [ ]:
import json
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# PlotConfig applies the hhlab style (resolves path from the installed package,
# avoiding font-cache issues that occur with relative style paths in notebooks)
from omero_screen_plots.config import CM_TO_INCHES, CONFIG  # noqa: F401
from omero_screen_plots.colors import COLOR

FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)
BENCHMARK_DIR = Path('.')

# Verify style is active
print('Font family:', plt.rcParams['font.family'])
print('Font sans-serif:', plt.rcParams['font.sans-serif'][:3])
print('Font size:', plt.rcParams['font.size'])

In [ ]:
# ---- Colours and labels --------------------------------------------------------

STAGE_COLORS = {
    'download_s':             COLOR.GREY.value,
    'nucleus_segmentation_s': COLOR.BLUE.value,
    'cell_segmentation_s':    COLOR.LIGHT_BLUE.value,
    'feature_extraction_s':   COLOR.YELLOW.value,
}
STAGE_NAMES = {
    'download_s':             'Download',
    'nucleus_segmentation_s': 'Nucleus seg.',
    'cell_segmentation_s':    'Cell seg.',
    'feature_extraction_s':   'Feature extraction',
}
STAGES = list(STAGE_COLORS.keys())

DEVICE_LABELS = {
    'cpu':                     'CPU',
    'mps':                     'MPS',
    'NVIDIA_A40':              'A40',
    'NVIDIA_GeForce_RTX_5090': 'RTX5090',
}

LABEL_ORDER = [
    'CPU (cp3)',
    'MPS (cp3)',
    'MPS (cp4)',
    'A40 (cp3)',
    'A40 (cp4)',
    'RTX5090 (cp3)',
    'RTX5090 (cp4)',
]


def _tick(label: str) -> str:
    """Convert 'MPS (cp3)' -> two-line tick label 'MPS\n(cp3)'."""
    return label.replace(' (', '\n(')


HW_COLORS = {
    'CPU':     COLOR.BLUE.value,
    'MPS':     COLOR.LIGHT_BLUE.value,
    'A40':     COLOR.YELLOW.value,
    'RTX5090': COLOR.PINK.value,
}


def _hw_color(label: str) -> str:
    for hw, color in HW_COLORS.items():
        if str(label).startswith(hw):
            return color
    return COLOR.GREY.value


# ---- Figure sizing -------------------------------------------------------------
FIG_H        = 5 * CM_TO_INCHES   # 5 cm height for all panels
FIG_W_WIDE   = 7 * CM_TO_INCHES   # wide panels (A, B, D, E)
FIG_W_NARROW = 4 * CM_TO_INCHES   # narrow panel (C) — 3 bars only


# ---- Grouped x-axis layout ----------------------------------------------------
BAR_GROUPS: list[list[str]] = [
    ['CPU (cp3)', 'MPS (cp3)', 'MPS (cp4)'],
    ['A40 (cp3)', 'A40 (cp4)'],
    ['RTX5090 (cp3)', 'RTX5090 (cp4)'],
]
GROUP_GAP = 0.4


def _group_x(groups: list[list[str]], gap: float = GROUP_GAP) -> dict[str, float]:
    """Map each config label to an x position, with inter-group gaps."""
    positions: dict[str, float] = {}
    pos = 0.0
    for i, group in enumerate(groups):
        if i > 0:
            pos += gap
        for label in group:
            positions[label] = pos
            pos += 1.0
    return positions


X_POS = _group_x(BAR_GROUPS)
X_MAX = max(X_POS.values())

GROUP_SEPARATORS: list[float] = [
    (X_POS[BAR_GROUPS[i][-1]] + X_POS[BAR_GROUPS[i + 1][0]]) / 2
    for i in range(len(BAR_GROUPS) - 1)
]


def _apply_group_style(ax: plt.Axes) -> None:
    """Set x-limits and draw subtle dotted separator lines between hardware groups."""
    ax.set_xlim(-0.5, X_MAX + 0.5)
    for sep in GROUP_SEPARATORS:
        ax.axvline(sep, color='#CCCCCC', linewidth=0.6, linestyle=':', zorder=1)


print('X positions:', X_POS)
print('Separators at:', GROUP_SEPARATORS)
print(f'Wide: {FIG_W_WIDE*2.54:.1f} cm  Narrow: {FIG_W_NARROW*2.54:.1f} cm  Height: {FIG_H*2.54:.1f} cm')


In [ ]:
# ---- Data loading --------------------------------------------------------------

def _parse_folder(name: str) -> dict[str, str]:
    """Extract device, cellpose version, and date from a benchmark folder name."""
    parts = name.split('_')
    return {
        'hostname': parts[0],
        'date':     parts[-1],
        'cellpose': parts[-2],
        'device':   '_'.join(parts[1:-2]),
    }


def _label(device: str, cellpose: str) -> str:
    hw = DEVICE_LABELS.get(device, device.replace('_', ' '))
    return f'{hw} ({cellpose})'


run_records:   list[dict] = []
image_records: list[dict] = []

for folder in sorted(BENCHMARK_DIR.iterdir()):
    if not folder.is_dir():
        continue
    if not any(f'_cp{v}_' in folder.name for v in ['3', '4']):
        continue

    parsed = _parse_folder(folder.name)
    lbl    = _label(parsed['device'], parsed['cellpose'])

    for json_path in sorted(folder.glob('benchmark_plate_*.json')):
        with open(json_path) as fh:
            data = json.load(fh)

        stgs = data['stages']
        run_records.append({
            'label':                  lbl,
            'device':                 parsed['device'],
            'cellpose':               parsed['cellpose'],
            'run_file':               json_path.name,
            'total_s':                stgs['total_s'],
            'metadata_parsing_s':     stgs.get('metadata_parsing', 0),
            'flatfield_correction_s': stgs.get('flatfield_correction', 0),
            'cell_cycle_analysis_s':  stgs.get('cell_cycle_analysis', 0),
            'save_results_s':         stgs.get('save_results', 0),
        })

        for img_idx, img in enumerate(data['per_image']):
            image_records.append({
                'label':                   lbl,
                'device':                  parsed['device'],
                'cellpose':                parsed['cellpose'],
                'run_file':               json_path.name,
                'image_idx':               img_idx,
                'image_id':                img['image_id'],
                'well':                    img['well'],
                'download_s':              img.get('download_s', 0),
                'nucleus_segmentation_s':  img.get('nucleus_segmentation_s', 0),
                'cell_segmentation_s':     img.get('cell_segmentation_s', 0),
                'feature_extraction_s':    img.get('feature_extraction_s', 0),
                'total_s':                 img['total_s'],
            })

cat       = pd.CategoricalDtype(categories=LABEL_ORDER, ordered=True)
df_runs   = pd.DataFrame(run_records)
df_images = pd.DataFrame(image_records)
df_runs['label']   = df_runs['label'].astype(cat)
df_images['label'] = df_images['label'].astype(cat)

print(f'Loaded {len(df_runs)} runs across {df_runs["label"].nunique()} configurations')
df_runs.groupby('label', observed=True)['total_s'].agg(['count', 'mean', 'std']).round(2)

In [ ]:
# ---- Preprocessing — shared aggregations used by multiple panels ---------------

# Exclude image_idx 0: first image per run carries GPU model-loading overhead
df_steady = df_images[df_images['image_idx'] > 0].copy()

# Per-run mean of steady-state stages (n=3 images per run)
df_run_means = (
    df_steady
    .groupby(['label', 'run_file'], observed=True)[STAGES]
    .mean()
    .reset_index()
)
df_run_means['total_image_s'] = df_run_means[STAGES].sum(axis=1)

# Mean +- SD across 10 runs  (Panels A, D)
_agg_cols = STAGES + ['total_image_s']
df_stats = (
    df_run_means
    .groupby('label', observed=True)[_agg_cols]
    .agg(['mean', 'std'])
    .reset_index()
)
df_stats.columns = ['label'] + [
    f'{c}_{s}' for c in _agg_cols for s in ['mean', 'std']
]
df_stats = df_stats.sort_values('label').reset_index(drop=True)

# Total run time stats  (Panel B)
run_stats = (
    df_runs
    .groupby('label', observed=True)['total_s']
    .agg(['mean', 'std'])
    .reset_index()
    .sort_values('label')
    .reset_index(drop=True)
)

print('Steady-state per-image means (s):')
df_stats[['label'] + [f'{s}_mean' for s in _agg_cols]].round(2)

## Panel A — Per-image processing time (steady-state)
Stacked bars show mean time per stage; error bars show SD of total per-image time across 10 runs (images 2–4 only).

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_W_WIDE, FIG_H))

x     = np.array([X_POS[str(lbl)] for lbl in df_stats['label']])
width = 0.85

bottoms = np.zeros(len(x))
for stage in STAGES:
    means = df_stats[f'{stage}_mean'].values
    ax.bar(x, means, width, bottom=bottoms,
           color=STAGE_COLORS[stage], label=STAGE_NAMES[stage], edgecolor='none')
    bottoms += means

totals = df_stats['total_image_s_mean'].values
errs   = df_stats['total_image_s_std'].values
ax.errorbar(x, totals, yerr=errs,
            fmt='none', color='black', capsize=2.5, linewidth=0.8, capthick=0.8, zorder=5)

for xi, t, e in zip(x, totals, errs):
    ax.text(xi, t + e + 0.5, f'{t:.1f}', ha='center', va='bottom', fontsize=5)

ax.set_xticks(x)
ax.set_xticklabels([_tick(str(lbl)) for lbl in df_stats['label']])
ax.set_ylabel('Time per image (s)')
ax.legend(frameon=True, loc='upper left')
ax.set_title(
    'Per-image processing time breakdown',
    loc='left', fontsize=8, fontweight='bold'
)

handles = [mpatches.Patch(color=STAGE_COLORS[s], label=STAGE_NAMES[s]) for s in STAGES]
ax.legend(handles=handles, loc='upper right', bbox_to_anchor=(1, 1), frameon=True)
_apply_group_style(ax)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'panel_a_per_image.pdf')
fig.savefig(FIGURES_DIR / 'panel_a_per_image.png', dpi=300)
plt.show()

## Panel B — Total run time
Full pipeline wall-clock time for a 4-image plate (includes flatfield correction, metadata parsing, cell cycle analysis). Individual runs shown as jittered dots.

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_W_WIDE, FIG_H))

x     = np.array([X_POS[str(lbl)] for lbl in run_stats['label']])
width = 0.85

bar_colors = [_hw_color(lbl) for lbl in run_stats['label']]
ax.bar(x, run_stats['mean'].values, width, color=bar_colors, edgecolor='none', zorder=2)
ax.errorbar(
    x, run_stats['mean'].values, yerr=run_stats['std'].values,
    fmt='none', color='black', capsize=2.5, linewidth=0.8, capthick=0.8, zorder=4,
)

rng        = np.random.default_rng(0)
label_to_x = {str(lbl): X_POS[str(lbl)] for lbl in run_stats['label'].values}
df_sorted  = df_runs.sort_values('label')
xi         = np.array([label_to_x[str(lbl)] for lbl in df_sorted['label']])
ax.scatter(
    xi + rng.uniform(-0.18, 0.18, len(xi)),
    df_sorted['total_s'].values,
    s=4, color='black', alpha=0.5, zorder=5, linewidths=0,
)

ax.set_xticks(x)
ax.set_xticklabels([_tick(str(lbl)) for lbl in run_stats['label']])
ax.set_ylabel('Total run time (s)')
ax.set_title(
    'Total pipeline time',
    loc='left', fontsize=8, fontweight='bold'
)

hw_patches = [mpatches.Patch(color=c, label=hw) for hw, c in HW_COLORS.items()]
ax.legend(handles=hw_patches, loc='upper right', frameon=True)
_apply_group_style(ax)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'panel_b_total_time.pdf')
fig.savefig(FIGURES_DIR / 'panel_b_total_time.png', dpi=300)
plt.show()


## Panel C — Cellpose cp4 vs cp3 fold-change
Ratio of steady-state segmentation time (nucleus + cell) for cp4 relative to cp3 on matched hardware. Values > 1 mean cp4 is slower. Error bars show propagated SEM.

In [ ]:
PAIRED_HW = ['MPS', 'A40', 'RTX5090']

df_seg = df_steady.copy()
df_seg['segmentation_s'] = df_seg['nucleus_segmentation_s'] + df_seg['cell_segmentation_s']

df_seg_run = (
    df_seg
    .groupby(['label', 'device', 'cellpose', 'run_file'], observed=True)['segmentation_s']
    .mean()
    .reset_index()
)
df_seg_run['hw'] = df_seg_run['label'].apply(lambda lbl: str(lbl).split(' (')[0])
df_seg_run = df_seg_run[df_seg_run['hw'].isin(PAIRED_HW)]

seg_stats = (
    df_seg_run
    .groupby(['hw', 'cellpose'])['segmentation_s']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)

hw_order = ['MPS', 'A40', 'RTX5090']
fc_rows  = []
for hw in hw_order:
    r3 = seg_stats[(seg_stats['hw'] == hw) & (seg_stats['cellpose'] == 'cp3')].iloc[0]
    r4 = seg_stats[(seg_stats['hw'] == hw) & (seg_stats['cellpose'] == 'cp4')].iloc[0]
    fc    = r4['mean'] / r3['mean']
    sem3  = r3['std'] / np.sqrt(r3['count'])
    sem4  = r4['std'] / np.sqrt(r4['count'])
    fc_err = fc * np.sqrt((sem4 / r4['mean'])**2 + (sem3 / r3['mean'])**2)
    fc_rows.append({'hw': hw, 'fold_change': fc, 'fc_err': fc_err,
                    'cp3_mean': r3['mean'], 'cp4_mean': r4['mean']})
df_fc = pd.DataFrame(fc_rows)

fig, ax = plt.subplots(figsize=(FIG_W_NARROW, FIG_H))

x          = np.arange(len(df_fc))
bar_colors = [HW_COLORS.get(hw, COLOR.GREY.value) for hw in df_fc['hw']]
ax.bar(x, df_fc['fold_change'], color=bar_colors, edgecolor='none', zorder=2)
ax.errorbar(
    x, df_fc['fold_change'], yerr=df_fc['fc_err'],
    fmt='none', color='black', capsize=2.5, linewidth=0.8, capthick=0.8, zorder=4,
)
ax.axhline(1.0, color='black', linewidth=0.75, linestyle='--', zorder=3)

for i, row in df_fc.iterrows():
    y_pos = row['fold_change'] + row['fc_err'] + 0.25
    ax.text(i, y_pos, f"{row['fold_change']:.1f}\u00d7",
            ha='center', va='bottom', fontsize=5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(df_fc['hw'].tolist())
ax.set_ylabel('Fold change  (cp4 / cp3  segmentation time)')
ax.set_title(
    'Cellpose cp4 vs cp3  |  nucleus + cell segmentation\n'
    'steady-state, mean \u00b1 propagated SEM, n=10',
    loc='left',
)
ax.set_xlim(-0.5, len(df_fc) - 0.5)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'panel_c_fold_change.svg')
fig.savefig(FIGURES_DIR / 'panel_c_fold_change.png', dpi=300)
plt.show()

print(df_fc[['hw', 'cp3_mean', 'cp4_mean', 'fold_change']].round(2).to_string(index=False))


## Panel D — Stage proportions (bottleneck analysis)
What fraction of steady-state per-image time is spent on each stage. A large download fraction indicates that network bandwidth — not GPU compute — is the limiting factor.

In [ ]:
stage_means = df_stats[[f'{s}_mean' for s in STAGES]].values
totals_row  = stage_means.sum(axis=1, keepdims=True)
fractions   = stage_means / totals_row

fig, ax = plt.subplots(figsize=(FIG_W_WIDE, FIG_H))

x     = np.array([X_POS[str(lbl)] for lbl in df_stats['label']])
width = 0.85

bottoms = np.zeros(len(x))
for j, stage in enumerate(STAGES):
    frac = fractions[:, j] * 100
    ax.bar(x, frac, width, bottom=bottoms,
           color=STAGE_COLORS[stage], label=STAGE_NAMES[stage], edgecolor='none')
    for xi, f, b in zip(x, frac, bottoms):
        if f > 10:
            ax.text(xi, b + f / 2, f'{f:.0f}%',
                    ha='center', va='center', fontsize=4.5, color='white', fontweight='bold')
    bottoms += frac

ax.set_xticks(x)
ax.set_xticklabels([_tick(str(lbl)) for lbl in df_stats['label']])
ax.set_ylabel('Fraction of per-image time (%)')
ax.set_title('Per-image time breakdown', loc='left', fontsize=8, fontweight='bold')
ax.set_ylim(0, 100)

handles = [mpatches.Patch(color=STAGE_COLORS[s], label=STAGE_NAMES[s]) for s in STAGES]
ax.legend(handles=handles, loc='upper right', frameon=True)
_apply_group_style(ax)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'panel_d_stage_proportions.pdf')
fig.savefig(FIGURES_DIR / 'panel_d_stage_proportions.png', dpi=300)
plt.show()

## Panel E — GPU model warm-up effect
Nucleus segmentation time for the first image per run vs steady-state mean (images 2–4). Log scale spans the full range. The warm-up spike reflects GPU model loading and is most severe for NVIDIA GPUs — relevant for small-plate or single-image workflows.

In [ ]:
df_first = (
    df_images[df_images['image_idx'] == 0]
    [['label', 'run_file', 'nucleus_segmentation_s']]
    .rename(columns={'nucleus_segmentation_s': 'first_s'})
)
df_ss = (
    df_images[df_images['image_idx'] > 0]
    .groupby(['label', 'run_file'], observed=True)['nucleus_segmentation_s']
    .mean()
    .reset_index()
    .rename(columns={'nucleus_segmentation_s': 'steady_s'})
)

df_warmup = df_first.merge(df_ss, on=['label', 'run_file'])
warmup_stats = (
    df_warmup
    .groupby('label', observed=True)[['first_s', 'steady_s']]
    .agg(['mean', 'std'])
    .reset_index()
)
warmup_stats.columns = ['label', 'first_mean', 'first_std', 'steady_mean', 'steady_std']
warmup_stats = warmup_stats.sort_values('label').reset_index(drop=True)
warmup_stats['warmup_factor'] = warmup_stats['first_mean'] / warmup_stats['steady_mean']

fig, ax = plt.subplots(figsize=(FIG_W_WIDE, FIG_H))

x     = np.array([X_POS[str(lbl)] for lbl in warmup_stats['label']])
bar_w = 0.35

ax.bar(x - bar_w / 2, warmup_stats['first_mean'], bar_w,
       color=COLOR.PINK.value, label='First image (warm-up)', edgecolor='none', zorder=2)
ax.errorbar(x - bar_w / 2, warmup_stats['first_mean'], yerr=warmup_stats['first_std'],
            fmt='none', color='black', capsize=2, linewidth=0.7, capthick=0.7, zorder=4)

ax.bar(x + bar_w / 2, warmup_stats['steady_mean'], bar_w,
       color=COLOR.BLUE.value, label='Steady-state (images 2\u20134)', edgecolor='none', zorder=2)
ax.errorbar(x + bar_w / 2, warmup_stats['steady_mean'], yerr=warmup_stats['steady_std'],
            fmt='none', color='black', capsize=2, linewidth=0.7, capthick=0.7, zorder=4)

for xi, row in zip(x, warmup_stats.itertuples()):
    if row.warmup_factor > 1.5:
        ax.text(xi - bar_w / 2,
                row.first_mean + row.first_std * 1.15,
                f'{row.warmup_factor:.0f}\u00d7',
                ha='center', va='bottom', fontsize=5,
                color=COLOR.PINK.value, fontweight='bold')

ax.set_yscale('log')
ax.set_ylim(0.1, 200)
ax.set_xticks(x)
ax.set_xticklabels([_tick(str(lbl)) for lbl in warmup_stats['label']])
ax.set_ylabel('Nucleus segmentation time (s, log scale)')
ax.set_title(
    'GPU model warm-up',
    loc='left', fontsize=8, fontweight='bold'
)
ax.legend(frameon=True, loc='upper right')
_apply_group_style(ax)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'panel_e_warmup.svg')
fig.savefig(FIGURES_DIR / 'panel_e_warmup.png', dpi=300)
plt.show()

print('Warm-up factors per configuration:')
print(warmup_stats[['label', 'first_mean', 'steady_mean', 'warmup_factor']].round(2).to_string(index=False))